# DDQN Channel Allocation Experiments

This notebook runs the DDQN-based enhancement on top of the CA-DRL environment and compares:

- CA-DRL (DQN)
- DDQN
- FCFS (paper baseline, fixed numbers)
- Random Access (paper baseline, fixed numbers)

using the metrics:

- Average delay (delay ratio)
- System throughput
- Cumulative reward

> Prerequisite: install dependencies from `requirements.txt` and run this notebook from the project root (`ca-ddqn`).

In [ ]:
import os
import json
from typing import List

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import sys
sys.path.append(os.getcwd())

from utils.config_loader import ExperimentConfig
from utils.reproducibility import ensure_reproducible_experiment
from environment.wireless_env import SatelliteEnvironment
from algorithms.ddqn_agent import DDQNAgent, DDQNTrainer
from algorithms.cadrl_agent import CADRLAgent, CADRLTrainer
from evaluation.metrics import EpisodeMetricsTracker, summarise_algorithm, attach_baseline_comparison
from evaluation.plots import plot_training_curves, plot_algorithm_comparison

In [ ]:
# Load experiment configuration and set seeds
cfg = ExperimentConfig.from_yaml()
seed_used = ensure_reproducible_experiment(cfg.random_seed)
seed_used

## (Optional) Train CA-DRL here

You can either:

- **Option A**: Load CA-DRL results produced by `training/train_cadrl.py`.
- **Option B**: Train CA-DRL directly in this notebook for a smaller number of episodes.

Set `RUN_CADRL_HERE` accordingly below.

In [ ]:
RUN_CADRL_HERE = False  # set True to train CA-DRL inline instead of loading JSON

In [ ]:
cadrl_summary = None

if RUN_CADRL_HERE:
    device = "cpu"
    env_c = SatelliteEnvironment(cfg)
    agent_c = CADRLAgent(env_c, cfg.training, device=device)
    trainer_c = CADRLTrainer(env_c, agent_c)

    num_episodes_c = 1000  # smaller interactive run; increase for more accurate reproduction
    tracker_c = EpisodeMetricsTracker(window_size=cfg.evaluation.metrics_window)
    rewards_c: List[float] = []
    delays_c: List[float] = []

    for ep in tqdm(range(num_episodes_c), desc="Training CA-DRL (inline)"):
        m = trainer_c.train_episode()
        r = m["episode_reward"]
        d = m["average_delay"]
        thr = m["throughput"]
        tracker_c.update(r, d, thr)
        rewards_c.append(r)
        delays_c.append(d)

    episodes_c = list(range(num_episodes_c))
    plot_training_curves(episodes_c, rewards_c, delays_c, save_path_prefix=os.path.join("results", "ca-drl", "cadrl_notebook_inline"))

    cadrl_summary = summarise_algorithm("CA-DRL", tracker_c.as_dict())
else:
    # Try to load summary from results created by training/train_cadrl.py
    cadrl_path = os.path.join("results", "ca-drl", "ca_drl_training_results.json")
    if os.path.exists(cadrl_path):
        with open(cadrl_path, "r") as f:
            data_c = json.load(f)
        cadrl_summary = data_c.get("summary", None)
    else:
        print("No CA-DRL summary found; run training/train_cadrl.py or set RUN_CADRL_HERE = True.")

cadrl_summary

## Train DDQN on the same environment

Now we train a DDQN agent using the exact same environment, state representation, action set, and reward structure as CA-DRL.

In [ ]:
device = "cpu"  # change to "cuda" if available

env_d = SatelliteEnvironment(cfg)
agent_d = DDQNAgent(env_d, cfg.training, device=device)
trainer_d = DDQNTrainer(env_d, agent_d)

num_episodes_d = 1000  # interactive run; increase for more thorough experiments
tracker_d = EpisodeMetricsTracker(window_size=cfg.evaluation.metrics_window)
rewards_d: List[float] = []
delays_d: List[float] = []

for ep in tqdm(range(num_episodes_d), desc="Training DDQN"):
    m = trainer_d.train_episode()
    r = m["episode_reward"]
    d = m["average_delay"]
    thr = m["throughput"]
    tracker_d.update(r, d, thr)
    rewards_d.append(r)
    delays_d.append(d)

len(rewards_d), np.mean(rewards_d[-100:]) if len(rewards_d) >= 100 else np.mean(rewards_d)

In [ ]:
# Plot DDQN training curves
episodes_d = list(range(len(rewards_d)))
plot_training_curves(episodes_d, rewards_d, delays_d, save_path_prefix=os.path.join("results", "ddqn", "ddqn_notebook"))

plt.figure(figsize=(6, 4))
plt.plot(episodes_d, rewards_d, label="DDQN reward")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(episodes_d, delays_d, label="DDQN delay", color="orange")
plt.xlabel("Episode")
plt.ylabel("Average delay")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Summarise DDQN performance
ddqn_summary = summarise_algorithm("DDQN", tracker_d.as_dict())
ddqn_summary

## Algorithm comparison: FCFS, Random, CA-DRL, DDQN

We now build a comparison table that uses:

- FCFS and Random Access **from the original CA-DRL paper (Table 4)**;
- CA-DRL statistics from this project (script or notebook);
- DDQN statistics from the previous cell.

In [ ]:
if cadrl_summary is None:
    print("Warning: CA-DRL summary is missing; comparison will include only DDQN + baselines.")
    cadrl_summary_effective = {"average_delay": 0.0, "average_reward": 0.0}
else:
    cadrl_summary_effective = cadrl_summary

comparison = attach_baseline_comparison(
    ca_drl=cadrl_summary_effective,
    ddqn=ddqn_summary,
    fcfs_delay=cfg.baselines.fcfs_delay,
    fcfs_reward=cfg.baselines.fcfs_reward,
    random_delay=cfg.baselines.random_delay,
    random_reward=cfg.baselines.random_reward,
)
comparison

In [ ]:
# Plot comparison bar charts (delay and reward)
plots_dir = os.path.join("results", "ddqn", "plots")
os.makedirs(plots_dir, exist_ok=True)
save_path = os.path.join(plots_dir, "algorithm_comparison.png")

plot_algorithm_comparison(comparison, save_path=save_path)
save_path